In [8]:
"""
Energy analysis for Job-Level Predictive GPU Thermal Management.

Processes:
    10s, 30s, 60s, 120s, 180s, 240s, 300s, 360s, 600s

Calculates:
    1. Predictive cooling energy proxy
    2. 40/60/80/100% stepped reference
    3. Always-100% reference
    4. Runtime-weighted energy savings

Important:
- Energy is a normalized fan/cooling energy proxy, NOT measured kWh.
- The 40/60/80/100% reference uses actual peak temperature and is an
  oracle/reference calculation, not a causal real-time controller.
- Prediction CSVs do not contain id_job, so test-row identity is reconstructed
  using the same train_test_split configuration used for model evaluation.
"""

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = r"C:\datacenter_project"

# Change this only if your runtime file has a different name.
RUNTIME_FILE = os.path.join(BASE_DIR, "runtime_per_job.csv")

OUTPUT_DIR = os.path.join(BASE_DIR, "energy_analysis")
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

WINDOWS = [10, 30, 60, 120, 180, 240, 300, 360, 600]

# Your folders use 10s, 30s, 60s, etc.
WINDOW_FOLDER_TEMPLATE = "{window}s"

# Dataset shown in your folders:
# model_a_early_10sec_vs_full_v4.csv
DATASET_FILENAME_TEMPLATE = "model_a_early_{window}sec_vs_full_v4.csv"

# Prediction files shown in your XGBOOST folders:
# model_a_10s_test_predictions.csv
#
# The code below does NOT depend on this exact filename. It searches
# for the prediction CSV in the correct {window}s\XGBOOST folder.

TEST_SIZE = 0.15
RANDOM_STATE = 42


# ============================================================
# 2. THERMAL REGIONS
# ============================================================

def temp_to_tier(temp):
    if pd.isna(temp):
        return np.nan

    if temp < 35:
        return "Low"
    elif temp < 45:
        return "Medium"
    elif temp < 71:
        return "High"
    else:
        return "Critical"


# ============================================================
# 3. COOLING STRATEGIES
# ============================================================

# Data-derived PWM mapping used by the project.
PREDICTIVE_PWM = {
    "Low": 30.0,
    "Medium": 30.0,
    "High": 47.5,
    "Critical": 100.0,
}

# Requested stepped reference.
STEPPED_PWM = {
    "Low": 40.0,
    "Medium": 60.0,
    "High": 80.0,
    "Critical": 100.0,
}

ALWAYS_100_PWM = 100.0


# ============================================================
# 4. FILE SEARCH
# ============================================================

def find_dataset_file(window):
    """
    Find:
        C:\\datacenter_project\\{window}s\\
            model_a_early_{window}sec_vs_full_v4.csv
    """

    folder = os.path.join(
        BASE_DIR,
        WINDOW_FOLDER_TEMPLATE.format(window=window)
    )

    filename = DATASET_FILENAME_TEMPLATE.format(window=window)

    candidates = [
        os.path.join(folder, filename),

        # Optional alternate organization.
        os.path.join(
            BASE_DIR,
            "04_Early_Window_Datasets",
            WINDOW_FOLDER_TEMPLATE.format(window=window),
            filename
        ),

        os.path.join(BASE_DIR, filename),
    ]

    for path in candidates:
        if os.path.isfile(path):
            return path

    # Last-resort recursive search.
    matches = glob.glob(
        os.path.join(BASE_DIR, "**", filename),
        recursive=True
    )

    if matches:
        return matches[0]

    return None


def find_prediction_file(window):
    """
    Find the prediction CSV in:

        C:\\datacenter_project\\{window}s\\XGBOOST\\

    Your screenshot shows filenames without a visible .csv extension
    because Windows Explorer is hiding extensions. The actual file is
    expected to be a CSV.

    We search for any CSV whose filename contains 'prediction'.
    """

    folder = os.path.join(
        BASE_DIR,
        WINDOW_FOLDER_TEMPLATE.format(window=window),
        "XGBOOST"
    )

    if not os.path.isdir(folder):
        print(f"Prediction folder not found: {folder}")
        return None

    csv_files = glob.glob(
        os.path.join(folder, "*.csv")
    )

    prediction_files = [
        path for path in csv_files
        if "prediction" in os.path.basename(path).lower()
    ]

    if not prediction_files:
        return None

    # Prefer the normal test-prediction output.
    preferred = [
        path for path in prediction_files
        if "test_predictions" in os.path.basename(path).lower()
    ]

    if preferred:
        return preferred[0]

    return prediction_files[0]


# ============================================================
# 5. RUNTIME DATA
# ============================================================

def load_runtime():

    if not os.path.isfile(RUNTIME_FILE):
        raise FileNotFoundError(
            f"Runtime file not found:\n{RUNTIME_FILE}\n\n"
            "Check the filename/path at the top of the script."
        )

    runtime = pd.read_csv(RUNTIME_FILE)

    required = ["id_job", "runtime_seconds"]

    missing = [
        col for col in required
        if col not in runtime.columns
    ]

    if missing:
        raise ValueError(
            f"Runtime file is missing columns: {missing}\n"
            f"Available columns: {list(runtime.columns)}"
        )

    runtime = runtime[required].copy()

    runtime["id_job"] = pd.to_numeric(
        runtime["id_job"],
        errors="coerce"
    )

    runtime["runtime_seconds"] = pd.to_numeric(
        runtime["runtime_seconds"],
        errors="coerce"
    )

    runtime = runtime.dropna(
        subset=["id_job", "runtime_seconds"]
    )

    runtime["id_job"] = runtime["id_job"].astype("int64")

    # One runtime record per job.
    runtime = (
        runtime
        .drop_duplicates(
            subset=["id_job"],
            keep="first"
        )
        .reset_index(drop=True)
    )

    return runtime


# ============================================================
# 6. RECONSTRUCT TEST ROWS
# ============================================================

def reconstruct_test_rows(dataset, prediction_df):

    # Your early-window dataset uses this target.
    target = "full_temperature_gpu_peak"

    if target not in dataset.columns:
        raise ValueError(
            "The early-window dataset does not contain "
            "'full_temperature_gpu_peak'.\n"
            f"Available columns:\n{list(dataset.columns)}"
        )

    # Reproduce exactly the split used by the prediction generation.
    indices = np.arange(len(dataset))

    _, test_idx = train_test_split(
        indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

    reconstructed = (
        dataset.iloc[test_idx]
        .copy()
        .reset_index(drop=True)
    )

    prediction_df = prediction_df.reset_index(drop=True)

    if len(reconstructed) != len(prediction_df):
        raise ValueError(
            "Test-row count mismatch.\n"
            f"Reconstructed: {len(reconstructed)}\n"
            f"Prediction CSV: {len(prediction_df)}"
        )

    # Verify that reconstructed rows correspond exactly to the
    # prediction CSV.
    reconstructed_actual = pd.to_numeric(
        reconstructed[target],
        errors="coerce"
    ).to_numpy()

    prediction_actual = pd.to_numeric(
        prediction_df["actual_peak_temp"],
        errors="coerce"
    ).to_numpy()

    difference = np.abs(
        reconstructed_actual - prediction_actual
    )

    max_difference = np.nanmax(difference)

    if (
        not np.isfinite(max_difference)
        or max_difference > 1e-8
    ):
        raise ValueError(
            "\nTEST ROW RECONSTRUCTION FAILED.\n"
            f"Maximum temperature difference: {max_difference}\n\n"
            "The prediction CSV and early-window dataset do not appear "
            "to use the same train/test split.\n"
            "Expected:\n"
            "    test_size = 0.15\n"
            "    random_state = 42"
        )

    print(
        "    Test-row verification: PASS "
        f"(max difference = {max_difference:.10f} C)"
    )

    # Attach model prediction information.
    reconstructed["actual_peak_temp"] = prediction_actual

    reconstructed["predicted_peak_temp"] = pd.to_numeric(
        prediction_df["predicted_peak_temp"],
        errors="coerce"
    ).to_numpy()

    reconstructed["prediction_error"] = pd.to_numeric(
        prediction_df["error"],
        errors="coerce"
    ).to_numpy()

    return reconstructed


# ============================================================
# 7. ENERGY PROXY
# ============================================================

def cooling_energy_proxy(pwm_pct, runtime_seconds):
    """
    Normalized fan/cooling energy proxy:

        E_proxy = (PWM / 100)^3 * runtime_seconds

    This is not measured electrical energy.
    """

    pwm_fraction = (
        np.asarray(pwm_pct, dtype=float) / 100.0
    )

    runtime = np.asarray(
        runtime_seconds,
        dtype=float
    )

    return (
        pwm_fraction ** 3
    ) * runtime


def add_energy_columns(df):

    # Thermal tiers.
    df["predicted_tier"] = (
        df["predicted_peak_temp"]
        .apply(temp_to_tier)
    )

    df["actual_tier"] = (
        df["actual_peak_temp"]
        .apply(temp_to_tier)
    )

    # PWM selected by predictive model.
    df["predictive_pwm_pct"] = (
        df["predicted_tier"]
        .map(PREDICTIVE_PWM)
    )

    # Reference PWM selected using actual peak temperature.
    df["stepped_reference_pwm_pct"] = (
        df["actual_tier"]
        .map(STEPPED_PWM)
    )

    # Always-100% reference.
    df["always_100_pwm_pct"] = ALWAYS_100_PWM

    # Energy proxy.
    df["predictive_energy_proxy"] = cooling_energy_proxy(
        df["predictive_pwm_pct"],
        df["runtime_seconds"]
    )

    df["stepped_reference_energy_proxy"] = cooling_energy_proxy(
        df["stepped_reference_pwm_pct"],
        df["runtime_seconds"]
    )

    df["always_100_energy_proxy"] = cooling_energy_proxy(
        df["always_100_pwm_pct"],
        df["runtime_seconds"]
    )

    return df


# ============================================================
# 8. WINDOW SUMMARY
# ============================================================

def summarize_window(df, window):

    predictive = df[
        "predictive_energy_proxy"
    ].sum()

    stepped = df[
        "stepped_reference_energy_proxy"
    ].sum()

    always100 = df[
        "always_100_energy_proxy"
    ].sum()

    saving_vs_100 = (
        (1.0 - predictive / always100) * 100.0
        if always100 > 0
        else np.nan
    )

    saving_vs_stepped = (
        (1.0 - predictive / stepped) * 100.0
        if stepped > 0
        else np.nan
    )

    return {
        "window_seconds": window,
        "test_job_gpu_records": len(df),
        "unique_jobs": df["id_job"].nunique(),
        "total_runtime_seconds": df[
            "runtime_seconds"
        ].sum(),

        "predictive_energy_proxy": predictive,
        "stepped_40_60_80_100_energy_proxy": stepped,
        "always_100_energy_proxy": always100,

        "saving_vs_always_100_pct": saving_vs_100,
        "saving_vs_stepped_pct": saving_vs_stepped,

        "mean_actual_peak_temp_C": df[
            "actual_peak_temp"
        ].mean(),

        "mean_predicted_peak_temp_C": df[
            "predicted_peak_temp"
        ].mean(),

        "MAE_C": df[
            "prediction_error"
        ].abs().mean(),

        "RMSE_C": np.sqrt(
            np.mean(
                df["prediction_error"] ** 2
            )
        ),
    }


# ============================================================
# 9. PROCESS ONE WINDOW
# ============================================================

def process_window(window, runtime):

    print("\n" + "=" * 70)
    print(f"PROCESSING {window}s")
    print("=" * 70)

    dataset_path = find_dataset_file(window)
    prediction_path = find_prediction_file(window)

    if dataset_path is None:
        raise FileNotFoundError(
            f"Could not find dataset for {window}s.\n"
            f"Expected:\n"
            f"  C:\\datacenter_project\\{window}s\\"
            f"model_a_early_{window}sec_vs_full_v4.csv"
        )

    if prediction_path is None:
        raise FileNotFoundError(
            f"Could not find prediction CSV for {window}s.\n"
            f"Expected folder:\n"
            f"  C:\\datacenter_project\\{window}s\\XGBOOST\\\n"
            f"Files found there must contain 'prediction' in the name."
        )

    print(f"Dataset:    {dataset_path}")
    print(f"Prediction: {prediction_path}")

    dataset = pd.read_csv(dataset_path)
    predictions = pd.read_csv(prediction_path)

    print(f"    Dataset rows: {len(dataset)}")
    print(f"    Prediction rows: {len(predictions)}")

    required_prediction_cols = [
        "actual_peak_temp",
        "predicted_peak_temp",
        "error",
    ]

    missing = [
        col for col in required_prediction_cols
        if col not in predictions.columns
    ]

    if missing:
        raise ValueError(
            f"Prediction file is missing: {missing}\n"
            f"Available columns: {list(predictions.columns)}"
        )

    # Reconstruct test rows and verify alignment.
    df = reconstruct_test_rows(
        dataset,
        predictions
    )

    # Find job ID.
    if "id_job" in df.columns:
        job_col = "id_job"
    elif "job_id" in df.columns:
        job_col = "job_id"
    else:
        raise ValueError(
            f"No job-ID column found in {window}s dataset.\n"
            f"Available columns: {list(df.columns)}"
        )

    if job_col != "id_job":
        df["id_job"] = df[job_col]

    df["id_job"] = pd.to_numeric(
        df["id_job"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["id_job"]
    ).copy()

    df["id_job"] = df["id_job"].astype("int64")

    # --------------------------------------------------------
    # Runtime merge
    # --------------------------------------------------------

    df = df.merge(
        runtime,
        on="id_job",
        how="left",
        validate="many_to_one"
    )

    missing_runtime = (
        df["runtime_seconds"].isna().sum()
    )

    if missing_runtime > 0:

        missing_jobs = (
            df.loc[
                df["runtime_seconds"].isna(),
                "id_job"
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            f"{window}s: {missing_runtime} records have "
            "no matching runtime.\n"
            f"Missing job IDs (first 20): "
            f"{missing_jobs[:20]}"
        )

    print(
        f"    Runtime merge: PASS "
        f"({len(df)} records)"
    )

    # --------------------------------------------------------
    # Energy
    # --------------------------------------------------------

    df = add_energy_columns(df)

    # --------------------------------------------------------
    # Save detailed data
    # --------------------------------------------------------

    detailed_path = os.path.join(
        OUTPUT_DIR,
        f"energy_detail_{window}s.csv"
    )

    preferred_columns = [
        "id_job",
        "Node",
        "node_id",
        "gpu_index",
        "runtime_seconds",
        "actual_peak_temp",
        "predicted_peak_temp",
        "prediction_error",
        "actual_tier",
        "predicted_tier",
        "predictive_pwm_pct",
        "stepped_reference_pwm_pct",
        "always_100_pwm_pct",
        "predictive_energy_proxy",
        "stepped_reference_energy_proxy",
        "always_100_energy_proxy",
    ]

    available = [
        col for col in preferred_columns
        if col in df.columns
    ]

    remaining = [
        col for col in df.columns
        if col not in available
    ]

    df[available + remaining].to_csv(
        detailed_path,
        index=False
    )

    print(
        f"    Detailed file: {detailed_path}"
    )

    summary = summarize_window(
        df,
        window
    )

    return summary, df


# ============================================================
# 10. PLOTS
# ============================================================

def make_plots(summary_df):

    x = summary_df["window_seconds"].to_numpy()

    # ========================================================
    # NORMALIZED COOLING ENERGY
    # ========================================================
    #
    # E_proxy = (PWM / 100)^3 * runtime_seconds
    #
    # This is a normalized cooling-energy proxy, NOT Joules
    # or kWh because actual fan electrical power is not measured.
    #
    # Normalize every strategy against the same always-100%
    # baseline:
    #
    #   Normalized Energy (%) =
    #       Strategy Energy / Always-100 Energy * 100
    #
    # Therefore:
    #   100% = same energy as continuous 100% cooling
    #   50%  = half of that baseline
    #   lower = less cooling energy

    summary_df["predictive_energy_pct"] = (
        summary_df["predictive_energy_proxy"]
        / summary_df["always_100_energy_proxy"]
        * 100.0
    )

    summary_df["stepped_energy_pct"] = (
        summary_df["stepped_40_60_80_100_energy_proxy"]
        / summary_df["always_100_energy_proxy"]
        * 100.0
    )

    summary_df["always_100_energy_pct"] = 100.0

    # ========================================================
    # ONE MAIN ENERGY COMPARISON GRAPH
    # ========================================================

    plt.figure(figsize=(11, 6.5))

    plt.plot(
        x,
        summary_df["predictive_energy_pct"],
        marker="o",
        linewidth=2,
        label="Predictive Controller"
    )

    plt.plot(
        x,
        summary_df["stepped_energy_pct"],
        marker="o",
        linewidth=2,
        label="40/60/80/100% Reference"
    )

    plt.axhline(
        100.0,
        linestyle="--",
        linewidth=2,
        label="Always 100% Baseline"
    )

    plt.xlabel("Prediction Window (seconds)")
    plt.ylabel(
        "Normalized Cooling Energy "
        "(% of Always-100% Baseline)"
    )
    plt.title(
        "Normalized Cooling Energy Across Prediction Windows"
    )

    plt.xticks(x)
    plt.ylim(bottom=0)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    energy_plot_path = os.path.join(
        FIGURE_DIR,
        "normalized_cooling_energy_comparison.png"
    )

    plt.savefig(
        energy_plot_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    # ========================================================
    # MODEL ACCURACY GRAPH
    # ========================================================
    #
    # Kept separate because MAE is not an energy quantity.

    plt.figure(figsize=(11, 6.5))

    plt.plot(
        x,
        summary_df["MAE_C"],
        marker="o",
        linewidth=2
    )

    plt.xlabel("Prediction Window (seconds)")
    plt.ylabel("MAE (°C)")
    plt.title("Prediction MAE Across Windows")

    plt.xticks(x)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    mae_plot_path = os.path.join(
        FIGURE_DIR,
        "prediction_MAE_across_windows.png"
    )

    plt.savefig(
        mae_plot_path,
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print("\nPlots saved:")
    print(f"  {energy_plot_path}")
    print(f"  {mae_plot_path}")

    return summary_df


# ============================================================
# 11. MAIN
# ============================================================

def main():

    print("=" * 70)
    print("JOB-LEVEL PREDICTIVE GPU THERMAL MANAGEMENT")
    print("ENERGY ANALYSIS")
    print("=" * 70)

    # --------------------------------------------------------
    # Runtime
    # --------------------------------------------------------

    print("\nLoading runtime data...")

    runtime = load_runtime()

    print(
        f"Runtime records: {len(runtime)}"
    )

    print(
        f"Unique jobs: "
        f"{runtime['id_job'].nunique()}"
    )

    # --------------------------------------------------------
    # Process every window
    # --------------------------------------------------------

    summaries = []

    for window in WINDOWS:

        summary, _ = process_window(
            window,
            runtime
        )

        summaries.append(summary)

    summary_df = (
        pd.DataFrame(summaries)
        .sort_values("window_seconds")
        .reset_index(drop=True)
    )

    # Create normalized energy metrics and plots before saving outputs.
    summary_df = make_plots(summary_df)

    # --------------------------------------------------------
    # Save CSV
    # --------------------------------------------------------

    csv_path = os.path.join(
        OUTPUT_DIR,
        "energy_summary_all_windows.csv"
    )

    summary_df.to_csv(
        csv_path,
        index=False
    )

    # --------------------------------------------------------
    # Save Excel
    # --------------------------------------------------------

    xlsx_path = os.path.join(
        OUTPUT_DIR,
        "energy_summary_all_windows.xlsx"
    )

    assumptions = pd.DataFrame({
        "Item": [
            "Predictive PWM - Low",
            "Predictive PWM - Medium",
            "Predictive PWM - High",
            "Predictive PWM - Critical",
            "Reference PWM - Low",
            "Reference PWM - Medium",
            "Reference PWM - High",
            "Reference PWM - Critical",
            "Always-100 PWM",
            "Fan power model",
            "Energy unit",
            "Low threshold",
            "Medium threshold",
            "High threshold",
            "Critical threshold",
            "Test size",
            "Random state",
        ],
        "Value": [
            PREDICTIVE_PWM["Low"],
            PREDICTIVE_PWM["Medium"],
            PREDICTIVE_PWM["High"],
            PREDICTIVE_PWM["Critical"],
            STEPPED_PWM["Low"],
            STEPPED_PWM["Medium"],
            STEPPED_PWM["High"],
            STEPPED_PWM["Critical"],
            ALWAYS_100_PWM,
            "(PWM/100)^3",
            "Normalized cooling-energy proxy; not Joules or kWh",
            "<35 C",
            "35 to <45 C",
            "45 to <71 C",
            ">=71 C",
            TEST_SIZE,
            RANDOM_STATE,
        ]
    })

    with pd.ExcelWriter(
        xlsx_path,
        engine="openpyxl"
    ) as writer:

        summary_df.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

        assumptions.to_excel(
            writer,
            sheet_name="Assumptions",
            index=False
        )

    # --------------------------------------------------------
    # Plots
    # --------------------------------------------------------

    make_plots(summary_df)

    # --------------------------------------------------------
    # Final output
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("ENERGY ANALYSIS COMPLETE")
    print("=" * 70)

    print("\nNormalized Cooling Energy Summary:")
    print(
        summary_df[
            [
                "window_seconds",
                "predictive_energy_pct",
                "stepped_energy_pct",
                "always_100_energy_pct",
                "saving_vs_always_100_pct",
                "saving_vs_stepped_pct",
                "MAE_C",
            ]
        ].to_string(index=False)
    )

    print("\nGenerated:")
    print(csv_path)
    print(xlsx_path)
    print(FIGURE_DIR)


if __name__ == "__main__":
    main()


JOB-LEVEL PREDICTIVE GPU THERMAL MANAGEMENT
ENERGY ANALYSIS

Loading runtime data...
Runtime records: 2508
Unique jobs: 2508

PROCESSING 10s
Dataset:    C:\datacenter_project\10s\model_a_early_10sec_vs_full_v4.csv
Prediction: C:\datacenter_project\10s\XGBOOST\model_a_10s_test_predictions.csv
    Dataset rows: 2739
    Prediction rows: 411
    Test-row verification: PASS (max difference = 0.0000000000 C)
    Runtime merge: PASS (411 records)
    Detailed file: C:\datacenter_project\energy_analysis\energy_detail_10s.csv

PROCESSING 30s
Dataset:    C:\datacenter_project\30s\model_a_early_30sec_vs_full_v4.csv
Prediction: C:\datacenter_project\30s\XGBOOST\model_a_30s_test_predictions.csv
    Dataset rows: 2739
    Prediction rows: 411
    Test-row verification: PASS (max difference = 0.0000000000 C)
    Runtime merge: PASS (411 records)
    Detailed file: C:\datacenter_project\energy_analysis\energy_detail_30s.csv

PROCESSING 60s
Dataset:    C:\datacenter_project\60s\model_a_early_60sec_vs_